# License Plate Detection — EDA
**CMPS 261 — Machine Learning Project**

This notebook explores the Car Plate Detection dataset before any modeling.
We will:
- Count and verify all images and annotations
- Understand image sizes and aspect ratios
- Visualize bounding box distributions
- Spot any issues in the dataset

In [ ]:
import os
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import random

# Paths
DATA_DIR   = '../data/archive'
IMG_DIR    = os.path.join(DATA_DIR, 'images')
ANN_DIR    = os.path.join(DATA_DIR, 'annotations')

print(f'Images      : {len(os.listdir(IMG_DIR))}')
print(f'Annotations : {len(os.listdir(ANN_DIR))}')

## 1. Parse All Annotations

In [ ]:
records = []

for xml_file in sorted(os.listdir(ANN_DIR)):
    if not xml_file.endswith('.xml'):
        continue
    tree = ET.parse(os.path.join(ANN_DIR, xml_file))
    root = tree.getroot()

    filename  = root.find('filename').text
    img_w     = int(root.find('size/width').text)
    img_h     = int(root.find('size/height').text)

    for obj in root.findall('object'):
        xmin = int(obj.find('bndbox/xmin').text)
        ymin = int(obj.find('bndbox/ymin').text)
        xmax = int(obj.find('bndbox/xmax').text)
        ymax = int(obj.find('bndbox/ymax').text)

        box_w  = xmax - xmin
        box_h  = ymax - ymin
        area   = box_w * box_h
        rel_w  = box_w / img_w   # relative width  (0–1)
        rel_h  = box_h / img_h   # relative height (0–1)

        records.append({
            'filename': filename,
            'img_w': img_w, 'img_h': img_h,
            'xmin': xmin, 'ymin': ymin, 'xmax': xmax, 'ymax': ymax,
            'box_w': box_w, 'box_h': box_h,
            'area': area,
            'rel_w': rel_w, 'rel_h': rel_h,
            'aspect_ratio': box_w / box_h if box_h > 0 else 0
        })

df = pd.DataFrame(records)
print(f'Total bounding boxes: {len(df)}')
df.head()

## 2. Image Size Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['img_w'], bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('Image Width Distribution')
axes[0].set_xlabel('Width (px)')
axes[0].set_ylabel('Count')

axes[1].hist(df['img_h'], bins=20, color='salmon', edgecolor='white')
axes[1].set_title('Image Height Distribution')
axes[1].set_xlabel('Height (px)')

plt.tight_layout()
plt.savefig('../results/image_size_distribution.png', dpi=150)
plt.show()

print(df[['img_w', 'img_h']].describe())

## 3. Bounding Box Size & Aspect Ratio

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df['rel_w'], bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('Relative Box Width')
axes[0].set_xlabel('Width / Image Width')

axes[1].hist(df['rel_h'], bins=20, color='salmon', edgecolor='white')
axes[1].set_title('Relative Box Height')
axes[1].set_xlabel('Height / Image Height')

axes[2].hist(df['aspect_ratio'], bins=20, color='mediumseagreen', edgecolor='white')
axes[2].set_title('Box Aspect Ratio (W/H)')
axes[2].set_xlabel('Aspect Ratio')

plt.tight_layout()
plt.savefig('../results/bbox_distributions.png', dpi=150)
plt.show()

print(df[['rel_w', 'rel_h', 'aspect_ratio']].describe())

## 4. Bounding Box Center Heatmap
Where do license plates tend to appear in the image?

In [ ]:
cx = ((df['xmin'] + df['xmax']) / 2) / df['img_w']
cy = ((df['ymin'] + df['ymax']) / 2) / df['img_h']

plt.figure(figsize=(6, 5))
plt.hist2d(cx, cy, bins=20, cmap='hot')
plt.colorbar(label='Count')
plt.title('License Plate Center Heatmap (normalized)')
plt.xlabel('Relative X')
plt.ylabel('Relative Y')
plt.gca().invert_yaxis()
plt.savefig('../results/bbox_center_heatmap.png', dpi=150)
plt.show()

## 5. Sample Images with Bounding Boxes

In [ ]:
sample_files = random.sample(df['filename'].unique().tolist(), 12)

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for ax, fname in zip(axes, sample_files):
    img_path = os.path.join(IMG_DIR, fname)
    img = Image.open(img_path).convert('RGB')
    ax.imshow(img)

    rows = df[df['filename'] == fname]
    for _, row in rows.iterrows():
        rect = patches.Rectangle(
            (row['xmin'], row['ymin']),
            row['box_w'], row['box_h'],
            linewidth=2, edgecolor='red', facecolor='none'
        )
        ax.add_patch(rect)

    ax.set_title(fname, fontsize=8)
    ax.axis('off')

plt.suptitle('Sample Images with Ground Truth Bounding Boxes', fontsize=13)
plt.tight_layout()
plt.savefig('../results/sample_images.png', dpi=150)
plt.show()

## 6. Dataset Summary

In [ ]:
print('=' * 45)
print('DATASET SUMMARY')
print('=' * 45)
print(f'Total images          : {df["filename"].nunique()}')
print(f'Total bounding boxes  : {len(df)}')
print(f'Images with 1 plate   : {(df.groupby("filename").size() == 1).sum()}')
print(f'Images with >1 plate  : {(df.groupby("filename").size() > 1).sum()}')
print(f'Avg image size        : {df["img_w"].mean():.0f} x {df["img_h"].mean():.0f} px')
print(f'Avg box size          : {df["box_w"].mean():.0f} x {df["box_h"].mean():.0f} px')
print(f'Avg relative box size : {df["rel_w"].mean():.2%} x {df["rel_h"].mean():.2%}')
print(f'Avg aspect ratio      : {df["aspect_ratio"].mean():.2f}')